# Alpamayo Triplane Lifting, step by step 🧭
### A guide to non-linear triplane tokenization for autonomous driving with `Halos-lift`

**Lifter name:** `alpamayo`

**Goal of this notebook:** understand how Alpamayo-R1 (NVIDIA, 2025) turns multi-camera images into
a compact fixed-size token sequence suitable for a VLA policy head, using three orthogonal feature
planes and a Mip-NeRF 360 non-linear scene parametrization.

Every cell runs on a CPU in seconds. No CUDA, no `mmcv`.

| Step | Question it answers |
|---|---|
| 1 | Triplane structure: Pxy, Pxz, Pyz |
| 2 | Why Mip-NeRF 360? (uniform vs non-linear grids) |
| 3 | `MipNeRF360Parametrizer` in code |
| 4 | How Alpamayo differs from TPVFormer |
| 5 | `AlpamayoReferencePoints` |
| 6 | Deformable attention and camera projections |
| 7 | `patchify_planes()` for VLA tokenization |
| 8 | Full forward pass |
| 9 | Train it and see it learn |
| 10 | Geometry verification (90° rig rotation) |
| 11 | Comparison: Alpamayo vs TPVFormer vs BEVFormer |
| 12 | Exercises and references |

---
**Key papers:**
- Ivanovic et al., *Efficient Multi-Camera Tokenization with Triplanes for End-to-End Driving*,  
  arXiv:2506.12251, 2025. (**The triplane tokenizer module**)
- Alpamayo-R1: *Bridging Reasoning and Action Prediction for Generalizable Autonomous Driving*,  
  arXiv:2511.00088, NVIDIA, 2025. (**The VLA that uses this tokenizer**)
- Barron et al., *Mip-NeRF 360: Unbounded Anti-Aliased Neural Radiance Fields*, CVPR 2022.  
  (**The scene parametrization used to space the grid cells**)
- Huang et al., *Tri-Perspective View for Vision-Based 3D Semantic Occupancy Prediction*, CVPR 2023.  
  (**TPVFormer — the architecture Alpamayo's tokenizer is based on**)

In [1]:
import math, time, os, sys
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

from lifting import GridSpec, Cameras
from lifting.data import SceneBatch, SyntheticSceneDataset
from lifting.lifters import LIFTERS
from lifting.lifters.alpamayo import (
    AlpamayoTriplaneLifter,
    AlpamayoReferencePoints,
    MipNeRF360Parametrizer,
)
from lifting.lifters.tpvformer import TPVFormerLifter, TPVReferencePoints, TPVPlanes

# Find lifting_diagrams.py
for _p in (".", "notebooks", os.path.join("..", "notebooks")):
    if os.path.exists(os.path.join(_p, "lifting_diagrams.py")):
        sys.path.insert(0, _p); break

torch.manual_seed(0)
plt.rcParams.update({"figure.dpi": 110, "image.interpolation": "nearest"})
print("torch", torch.__version__)
print("alpamayo registered:", "alpamayo" in LIFTERS)

torch 2.14.1
alpamayo registered: True


## Step 1: The triplane structure — Pxy, Pxz, Pyz

Alpamayo's tokenizer shares its basic **three-plane** structure with TPVFormer:
a 3D scene is represented by three orthogonal feature maps that "shadow" the scene from three directions.

| Plane | Shape | Projection direction | What it captures |
|---|---|---|---|
| **Pxy** (BEV) | `(B, C, X, Y)` | top-down | lane layout, object positions |
| **Pxz** (side) | `(B, C, X, Z)` | left-right | height distribution along forward axis |
| **Pyz** (front) | `(B, C, Y, Z)` | forward-back | height distribution along lateral axis |

To look up features at 3D point `(x, y, z)` you simply **sum the three lookups**:

$$f(x, y, z) = f_{xy}(x, y) + f_{xz}(x, z) + f_{yz}(y, z)$$

The key difference from TPVFormer: the grid that tiles each plane is **non-linear**, with denser cells near the ego
vehicle and coarser cells at the edges. This is the Mip-NeRF 360 idea, which we study in Step 2.

### How it compares

| method | planes | scene parametrization | downstream use |
|---|---|---|---|
| BEVFormer | Pxy only | uniform | BEV perception |
| TPVFormer | Pxy + Pxz + Pyz | uniform | 3D occupancy |
| **Alpamayo** | **Pxy + Pxz + Pyz** | **Mip-NeRF 360 non-linear** | **VLA policy (planning)** |

In [2]:
# 3D diagram: three orthogonal planes intersecting the scene bounding box
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection='3d')

xlo, xhi = -16, 16
ylo, yhi = -16, 16
zlo, zhi = -1,  3

# Pxy: z = 0 (BEV)
xx, yy = np.meshgrid([xlo, xhi], [ylo, yhi])
ax.plot_surface(xx, yy, np.zeros_like(xx), alpha=0.25, color='dodgerblue')
ax.text(18, 0, 0, 'Pxy\n(BEV)', color='dodgerblue', fontsize=9, fontweight='bold')

# Pxz: y = 0
xx2, zz2 = np.meshgrid([xlo, xhi], [zlo, zhi])
ax.plot_surface(xx2, np.zeros_like(xx2), zz2, alpha=0.25, color='orangered')
ax.text(18, 0, 3.5, 'Pxz\n(side)', color='orangered', fontsize=9, fontweight='bold')

# Pyz: x = 0
yy3, zz3 = np.meshgrid([ylo, yhi], [zlo, zhi])
ax.plot_surface(np.zeros_like(yy3), yy3, zz3, alpha=0.25, color='mediumseagreen')
ax.text(0, 18, 3.5, 'Pyz\n(front)', color='mediumseagreen', fontsize=9, fontweight='bold')

# Ego vehicle
ax.scatter([0], [0], [0], c='gold', s=200, zorder=10, label='ego')

ax.set_xlabel('x (forward, m)'); ax.set_ylabel('y (left, m)'); ax.set_zlabel('z (up, m)')
ax.set_title('Alpamayo: three orthogonal feature planes', pad=12)
ax.legend()
plt.tight_layout()
plt.show()

/var/folders/_q/55b37d8s6sq3xq4j5ll92c400000gn/T/ipykernel_5300/2872704482.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 2: Why Mip-NeRF 360? The non-linear scene parametrization

A **uniform** grid allocates the same number of cells to every metre of the scene.
But far-away objects subtend only a few pixels — the image resolution drops with distance.
Giving them the same number of cells as nearby objects wastes memory and receptive field.

**Mip-NeRF 360** (Barron et al. 2022) solves this in novel-view synthesis by dividing the scene into:
- **Inner region**: a bounded box around the origin, rendered at full resolution.
- **Outer region**: the unbounded remainder, mapped through a non-linear contraction that folds
  the infinite exterior back into a finite number of cells.

Ivanovic et al. 2025 adopt this idea for the triplane grid:

$$\text{cell}_i = \begin{cases}
  \text{evenly spaced in } [-r_{\text{in}},\, +r_{\text{in}}] & \text{if } i < S_{\text{in}} \\
  \text{coarsely spaced beyond } r_{\text{in}} & \text{otherwise}
\end{cases}$$

With `Sx=Sy=96` and `inner_frac=0.75`, 72 cells are in the inner zone and 24 are in the outer zone.
The result: **much denser coverage within ~36 m** (typical highway driving range) and coarse coverage beyond.

This is implemented in `MipNeRF360Parametrizer.cell_centers_m()`.

In [3]:
# Compare uniform vs Mip-NeRF 360 cell spacing on one axis (32 cells, 32 m extent)
total_cells = 32
extent = 32.0

uniform = torch.linspace(-extent/2 + extent/total_cells/2,
                          extent/2 - extent/total_cells/2, total_cells)

param = MipNeRF360Parametrizer(total=total_cells, inner_frac=0.75, total_extent_m=extent)
nonlinear = param.cell_centers_m()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3))

ax1.scatter(uniform.numpy(),  np.zeros(total_cells), s=30, alpha=0.8, label='uniform')
ax1.scatter(nonlinear.numpy(), np.ones(total_cells)*0.3, s=30, alpha=0.8, color='orangered', label='Mip-NeRF 360')
ax1.set_xlabel('Metric position (m)'); ax1.set_title('Cell centre positions on one axis')
ax1.set_yticks([]); ax1.legend()
r_in = param.inner_radius_m   # metric half-width of the linear inner zone
ax1.axvspan(-r_in, r_in, alpha=0.07, color='green', label='inner zone')

# Spacing (gap between consecutive cells)
ax2.plot(uniform[:-1].numpy(), uniform[1:].numpy() - uniform[:-1].numpy(),
         label='uniform spacing', marker='o', ms=4)
ax2.plot(nonlinear[:-1].numpy(), nonlinear[1:].numpy() - nonlinear[:-1].numpy(),
         label='Mip-NeRF 360 spacing', marker='s', ms=4, color='orangered')
ax2.axvspan(-r_in, r_in, alpha=0.07, color='green')
ax2.set_xlabel('Metric position (m)'); ax2.set_ylabel('Cell width (m)')
ax2.set_title('Cell widths: small = high resolution'); ax2.legend()

plt.tight_layout()
plt.show()

widths = param.cell_edges_m().diff()          # metric width of every cell
side = param.n_outer // 2                         # outer cells on each side
inner_w = widths[side:side + param.n_inner]
outer_w = torch.cat([widths[:side], widths[side + param.n_inner:]])
print(f"\nInner zone: {param.n_inner} cells covering {inner_w.sum():.1f} m  ({inner_w.mean():.2f} m/cell; uniform grid: {extent / total_cells:.2f})")
print(f"Outer zone: {param.n_outer} cells covering {outer_w.sum():.1f} m  "
      f"({outer_w.min():.2f} -> {outer_w.max():.2f} m/cell, widening with range)")



Inner zone: 24 cells covering 21.3 m  (0.89 m/cell; uniform grid: 1.00)
Outer zone: 8 cells covering 10.7 m  (0.97 -> 1.78 m/cell, widening with range)


/var/folders/_q/55b37d8s6sq3xq4j5ll92c400000gn/T/ipykernel_5300/3272584372.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 3: `MipNeRF360Parametrizer` in detail

The parametrizer takes three arguments:
- `total`: total number of cells on this axis.
- `inner_frac`: fraction of cells that form the inner high-res region.
- `total_extent_m`: total metric span in metres.

And returns `cell_centers_m()`: the metric position of each cell centre, monotonically sorted.

In [4]:
p = MipNeRF360Parametrizer(total=16, inner_frac=0.75, total_extent_m=32.0)
centers = p.cell_centers_m()
side = p.n_outer // 2
print(f"inner zone: {p.n_inner} cells within ±{p.inner_radius_m:.2f} m; {side} outer cells per side")
print("Cell centres (m):")
for i, c in enumerate(centers):
    zone = "inner" if side <= i < side + p.n_inner else "outer"
    print(f"  cell {i:2d}: {c:+7.2f} m  ({zone})")
print()
edges = p.cell_edges_m()
print("Sorted (monotone):", (centers[1:] > centers[:-1]).all().item())
print(f"Cell edges span: {edges[0].item():+.1f} m .. {edges[-1].item():+.1f} m  (= the full 32 m extent)")


inner zone: 12 cells within ±10.67 m; 2 outer cells per side
Cell centres (m):
  cell  0:  -14.22 m  (outer)
  cell  1:  -11.64 m  (outer)
  cell  2:   -9.78 m  (inner)
  cell  3:   -8.00 m  (inner)
  cell  4:   -6.22 m  (inner)
  cell  5:   -4.44 m  (inner)
  cell  6:   -2.67 m  (inner)
  cell  7:   -0.89 m  (inner)
  cell  8:   +0.89 m  (inner)
  cell  9:   +2.67 m  (inner)
  cell 10:   +4.44 m  (inner)
  cell 11:   +6.22 m  (inner)
  cell 12:   +8.00 m  (inner)
  cell 13:   +9.78 m  (inner)
  cell 14:  +11.64 m  (outer)
  cell 15:  +14.22 m  (outer)

Sorted (monotone): True
Cell edges span: -16.0 m .. +16.0 m  (= the full 32 m extent)


In [5]:
# Show for all three axes of a realistic grid (96, 96, 48)
grid_large = GridSpec(
    x_bounds=(-51.2, 51.2), y_bounds=(-51.2, 51.2), z_bounds=(-5.0, 3.0),
    resolution=(96, 96, 48),
)

fig, axes = plt.subplots(1, 3, figsize=(14, 3))
for axis_idx, (ax, name) in enumerate(zip(axes, ["x (forward)", "y (lateral)", "z (up)"])):
    p_ax = MipNeRF360Parametrizer(
        total=grid_large.resolution[axis_idx],
        inner_frac=0.75,
        total_extent_m=float(grid_large.bounds[axis_idx][1] - grid_large.bounds[axis_idx][0]),
    )
    cs = p_ax.cell_centers_m().numpy()
    ax.scatter(cs, np.zeros_like(cs), s=10, alpha=0.7)
    ax.set_xlabel('m'); ax.set_title(f'{name}  ({len(cs)} cells)')
    ax.set_yticks([])

plt.suptitle('Mip-NeRF 360 cell centres for Alpamayo-sized grid (96×96×48)', y=1.03)
plt.tight_layout()
plt.show()

/var/folders/_q/55b37d8s6sq3xq4j5ll92c400000gn/T/ipykernel_5300/887952986.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 4: How Alpamayo differs from TPVFormer

The two methods share the same transformer encoder architecture (`TPVFormerLayer` with its
deformable cross-attention and cross-view hybrid attention). The difference is exactly one component:
**where the reference points are placed** (i.e. which 3D locations each query looks at).

```
TPVFormer:                  Alpamayo:

grid.line_points()          MipNeRF360Parametrizer.cell_centers_m()
(uniform cell centres)      (non-linear: denser near ego)
        │                           │
        ▼                           ▼
TPVReferencePoints          AlpamayoReferencePoints
        │                           │
        ▼                           ▼
TPVFormerLayer (same)       TPVFormerLayer (same)
        │                           │
        ▼                           ▼
     TPVPlanes              TPVPlanes + patchify_planes()
        │                           │
        ▼                           ▼
  voxel/BEV output          VLA token sequence
```

Additionally, Alpamayo adds `patchify_planes()`, which folds the three planes into a flat token
sequence to feed the VLA policy head (Step 7).

In [6]:
grid = GridSpec(resolution=(32, 32, 4))

# Uniform reference points (TPVFormer)
tpv_refs = TPVReferencePoints(grid, num_anchors=(4, 8, 8))

# Mip-NeRF 360 reference points (Alpamayo)
alp_refs = AlpamayoReferencePoints(grid, num_anchors=(4, 8, 8), inner_frac=0.75)

print("Plane shapes (same for both):", alp_refs.plane_shapes)
print()

# Compare anchor positions for the xy plane, query at cell index 20,20
q = 20 * 32 + 20
tpv_anc = tpv_refs.anchor_points(0)
alp_anc = alp_refs.anchor_points(0)
print(f"Anchor points for xy query #{q} (cell 20,20 in a 32x32 grid):")
print("  TPVFormer (uniform)    :", tpv_anc[q].tolist())
print("  Alpamayo  (non-linear) :", alp_anc[q].tolist())
print("\nNote: x,y positions differ; z (the normal axis pillar) is still uniform")

Plane shapes (same for both): [(32, 32), (32, 4), (32, 4)]

Anchor points for xy query #660 (cell 20,20 in a 32x32 grid):
  TPVFormer (uniform)    : [[4.5, 4.5, -0.5], [4.5, 4.5, 0.5], [4.5, 4.5, 1.5], [4.5, 4.5, 2.5]]
  Alpamayo  (non-linear) : [[4.0, 4.0, -0.5], [4.0, 4.0, 0.5], [4.0, 4.0, 1.5], [4.0, 4.0, 2.5]]

Note: x,y positions differ; z (the normal axis pillar) is still uniform


In [7]:
# Show the x positions of query 0..31 (the first row) for both methods
tpv_x = tpv_anc[:32, 0, 0].numpy()   # x position of first anchor, first anchor set
alp_x = alp_anc[:32, 0, 0].numpy()

plt.figure(figsize=(9, 3.5))
plt.plot(np.arange(32), tpv_x, marker='o', ms=5, label='TPVFormer (uniform)')
plt.plot(np.arange(32), alp_x, marker='s', ms=5, label='Alpamayo (Mip-NeRF 360)')
plt.xlabel('Query index (x-axis of xy plane)')
plt.ylabel('Metric x position (m)')
plt.title('Query x positions: uniform vs non-linear grid')
plt.axhspan(-12, 12, alpha=0.07, color='green', label='inner 75% zone')
plt.legend(); plt.tight_layout(); plt.show()

/var/folders/_q/55b37d8s6sq3xq4j5ll92c400000gn/T/ipykernel_5300/4137786160.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.legend(); plt.tight_layout(); plt.show()


## Step 5: `AlpamayoReferencePoints` — where each query looks

Exactly like `TPVReferencePoints`, but with non-linear coordinates.

Key API:
- `anchor_points(plane)` → `(Q, A, 3)` metric 3D locations of all anchors for each query on `plane`.
- `camera_reference_points(cameras)` → per-plane list of `(B, N, Q, A, 2)` projected image coordinates.
- `hybrid_reference_points()` → `(Q_total, 3, 2)` cross-view attention locations (normalised, same as TPVFormer).

In [8]:
# Load data
data = SyntheticSceneDataset(256, grid, seed=42)
batch = SceneBatch.collate([data[i] for i in range(4)])
cams = batch.cameras

alp_refs = AlpamayoReferencePoints(grid, num_anchors=(4, 8, 8), inner_frac=0.75)

cam_refs = alp_refs.camera_reference_points(cams)
for p, (uv, vis) in enumerate(cam_refs):
    print(f"plane {p}: uv {tuple(uv.shape)} = (B, N, Q, A, 2),  visible {tuple(vis.shape)}")

hyb = alp_refs.hybrid_reference_points()
print("\nhybrid reference:", tuple(hyb.shape), "= (Q_total, 3, 2)")

plane 0: uv (4, 4, 1024, 4, 2) = (B, N, Q, A, 2),  visible (4, 4, 1024, 4)
plane 1: uv (4, 4, 128, 8, 2) = (B, N, Q, A, 2),  visible (4, 4, 128, 8)
plane 2: uv (4, 4, 128, 8, 2) = (B, N, Q, A, 2),  visible (4, 4, 128, 8)

hybrid reference: (1280, 3, 2) = (Q_total, 3, 2)


In [9]:
# Visualise reference points on camera images
b, cam_idx = 0, 0
H, W = cams.image_size
fig, ax = plt.subplots(1, 3, figsize=(15, 3.4))
for p, name in enumerate(["Pxy: pillars (z)", "Pxz: lines (y)", "Pyz: lines (x)"]):
    uv, vis = cam_refs[p]
    pts = uv[b, cam_idx][vis[b, cam_idx]] * torch.tensor([W, H])
    ax[p].imshow(batch.images[b, cam_idx].permute(1, 2, 0))
    ax[p].scatter(pts[:, 0].numpy(), pts[:, 1].numpy(), s=4, c='yellow', alpha=0.6)
    n_visible = vis[b, cam_idx].any(-1).sum().item()
    ax[p].set_title(f"{name}\n{n_visible} queries see cam {cam_idx}")
    ax[p].axis('off')

plt.suptitle('Alpamayo reference points projected onto camera 0', y=1.02)
plt.tight_layout()
plt.show()

/var/folders/_q/55b37d8s6sq3xq4j5ll92c400000gn/T/ipykernel_5300/2783906668.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 6: Deformable attention into camera features

Alpamayo uses the exact same `TPVFormerLayer` as TPVFormer:

```
tokens ──► cross-view hybrid attention  (planes talk to each other)
       ──► image cross-attention         (each plane attends to cameras, via ref. points)
       ──► feed-forward network
```

The only difference that reaches this layer is the `camera_reference_points()` it receives, which
are now Mip-NeRF 360-spaced rather than uniform.

See `tpv_lifting_step_by_step.ipynb` Steps 7–9 for a full walkthrough of how `SpatialCrossAttention`
and `CrossViewHybridAttention` work. Here we verify the shape contract.

In [10]:
from lifting.lifters.tpvformer.tpvformer_layer import TPVFormerLayer
from lifting.lifters.pyramid import flatten_pyramid
from lifting.encoders import SimpleConvEncoder

# Encode images
encoder = SimpleConvEncoder(out_channels=32, num_levels=1)
b, n = batch.images.shape[:2]
with torch.no_grad():
    feats = [f.view(b, n, *f.shape[1:])
             for f in encoder(batch.images.flatten(0, 1))]

print("Image features per level:")
for i, f in enumerate(feats):
    print(f"  level {i}: {tuple(f.shape)}")

# One TPVFormer layer (the same module reused by Alpamayo)
layer = TPVFormerLayer(embed_dims=32, num_heads=4, num_levels=1, num_anchors=(4, 8, 8))

plane_shapes = alp_refs.plane_shapes
Qtot = sum(h * w for h, w in plane_shapes)
query = torch.randn(b, Qtot, 32)
pos   = torch.randn(b, Qtot, 32)

values, shapes = flatten_pyramid(feats, torch.zeros(1, 32))
hyb_expanded = hyb[None].expand(b, -1, -1, -1)
cam_refs_this = alp_refs.camera_reference_points(cams)

with torch.no_grad():
    out = layer(query, pos, plane_shapes, hyb_expanded, values, shapes, cam_refs_this)

print(f"\nLayer: in {tuple(query.shape)} → out {tuple(out.shape)}  (shape preserved)")

Image features per level:
  level 0: (4, 4, 32, 12, 24)

Layer: in (4, 1280, 32) → out (4, 1280, 32)  (shape preserved)


## Step 7: `patchify_planes()` — compact VLA token sequence

BEV tokenization for a VLA policy head needs a **fixed, compact** token count.
Alpamayo-R1 achieves this by patchifying the three planes:

1. Divide each plane `(B, C, H, W)` into non-overlapping `p × p` patches.
2. Flatten each patch to a `C·p²` vector.
3. Concatenate all three planes along the token dimension.

For the paper's dimensions `Sx=Sy=96, Sz=48, patch_size=8`:

| Plane | Grid | Patches (p=8) | Tokens |
|---|---|---|---|
| Pxy | 96 × 96 | (96/8)² = 144 | 144 |
| Pxz | 96 × 48 | (96/8)×(48/8) = 72 | 72 |
| Pyz | 96 × 48 | 72 | 72 |
| **Total** | | | **288** |

Per-camera patch tokenization (6 cameras, 448×800 image, `patch_size=14` ViT-L) gives
~`6 × (32×57) ≈ 10944` tokens. **288 is 97% fewer.**

Even with smaller cameras like the synthetic rig (4 cameras, 48×96, patch 16), the comparison
is clear:

In [11]:
# Instantiate the full Alpamayo lifter and run patchify_planes
lifter = AlpamayoTriplaneLifter(
    grid=grid,
    in_channels=32,
    out_channels=32,
    num_layers=2,
    num_heads=4,
    inner_frac=0.75,
)

with torch.no_grad():
    planes = lifter.lift_planes(feats, cams)
    tokens = lifter.patchify_planes(planes, patch_size=4)

print("plane shapes:")
print(f"  xy: {tuple(planes.xy.shape)}")
print(f"  xz: {tuple(planes.xz.shape)}")
print(f"  yz: {tuple(planes.yz.shape)}")
print()
print(f"patchify_planes (patch=4): {tuple(tokens.shape)}")
n_xy  = (32 // 4) ** 2
n_xz  = (32 // 4) * (4 // 4)
n_yz  = (32 // 4) * (4 // 4)
print(f"  = {n_xy} (Pxy) + {n_xz} (Pxz) + {n_yz} (Pyz) = {n_xy + n_xz + n_yz} tokens")

# Compare with per-camera patch tokens
n_cam = 4; h_cam = 48 // 4; w_cam = 96 // 4
per_camera_tokens = n_cam * h_cam * w_cam
print(f"\nPer-camera patch tokens (4 cams, 48×96, p=4): {per_camera_tokens}")
print(f"Triplane tokens (p=4):                         {n_xy + n_xz + n_yz}")
print(f"Reduction:                                      {100*(1 - (n_xy+n_xz+n_yz)/per_camera_tokens):.0f}%")

plane shapes:
  xy: (4, 32, 32, 32)
  xz: (4, 32, 32, 4)
  yz: (4, 32, 32, 4)

patchify_planes (patch=4): (4, 80, 512)
  = 64 (Pxy) + 8 (Pxz) + 8 (Pyz) = 80 tokens

Per-camera patch tokens (4 cams, 48×96, p=4): 1152
Triplane tokens (p=4):                         80
Reduction:                                      93%


In [12]:
# Visualise the three planes feature magnitude (like a fingerprint of the scene)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, (name, plane) in zip(axes, [("Pxy (BEV)", planes.xy), ("Pxz (side)", planes.xz), ("Pyz (front)", planes.yz)]):
    im = ax.imshow(plane[0].norm(dim=0).detach().numpy(), cmap='magma', aspect='auto')
    ax.set_title(f"{name}\n|feature|")
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle('AlpamayoTriplaneLifter — plane feature magnitudes (untrained)', y=1.02)
plt.tight_layout()
plt.show()

/var/folders/_q/55b37d8s6sq3xq4j5ll92c400000gn/T/ipykernel_5300/57484233.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 8: Full forward pass

`AlpamayoTriplaneLifter.forward()` returns a BEV map, exactly like every other lifter in Halos-lift:

```
features  (B, N, C, H, W)   ─► lift_planes() ─► TPVPlanes(xy, xz, yz)
cameras   (B, N, ...)                             │ .to_voxels()  (sum of three planes)
                                                  ▼
                                            HeightCompressor
                                                  │
                                                  ▼
                                            BEV map (B, out_channels, X, Y)
```

This allows Alpamayo to be benchmarked head-to-head against BEVFormer, Lift-Splat, etc. in the
standard Halos-lift BEV segmentation task.

In [13]:
with torch.no_grad():
    bev = lifter(feats, cams)

print("Input images  :", tuple(batch.images.shape))
print("Image features:", tuple(feats[0].shape), "(B, N, C, H, W)")
print("BEV output    :", tuple(bev.shape), "= (B, out_channels, X, Y)")
print()
print("Lifter parameters:", f"{sum(p.numel() for p in lifter.parameters()):,}")

# Compare with TPVFormer on the same grid
tpv_lifter = TPVFormerLifter(grid, in_channels=32, out_channels=32, num_layers=2)
print("TPVFormer parameters:", f"{sum(p.numel() for p in tpv_lifter.parameters()):,}")
print("(Same — only the reference point computation differs)")

Input images  : (4, 4, 3, 48, 96)
Image features: (4, 4, 32, 12, 24) (B, N, C, H, W)
BEV output    : (4, 32, 32, 32) = (B, out_channels, X, Y)

Lifter parameters: 186,784
TPVFormer parameters: 186,784
(Same — only the reference point computation differs)


## Step 9: Train it and watch it learn

We train on the synthetic BEV segmentation task, identical to the other notebooks.
The metric is **mIoU over foreground classes** (vehicle + pedestrian).

In [14]:
from lifting.training import Trainer, BEVSegmentationTask
from lifting.encoders import SimpleConvEncoder
from lifting.models import build_bev_model

torch.manual_seed(0)

def make_batches(lo, hi, bs=4):
    return [SceneBatch.collate([data[i + j] for j in range(bs)]) for i in range(lo, hi, bs)]

train_batches = make_batches(0, 192)
test_batches  = make_batches(192, 256)

model = build_bev_model("alpamayo", grid, num_classes=3, channels=32)
trainer = Trainer(model, BEVSegmentationTask(num_classes=3))

print("mIoU before training:", round(trainer.evaluate(test_batches).mean_foreground(), 3))

t0 = time.time()
trainer.fit(train_batches, steps=250)
print(f"Trained 250 steps in {time.time() - t0:.0f}s")
print("mIoU after training :", round(trainer.evaluate(test_batches).mean_foreground(), 3))

mIoU before training: 0.005


Trained 250 steps in 27s


mIoU after training : 0.496


In [15]:
losses = trainer.history.losses
plt.figure(figsize=(7, 3))
plt.plot(losses, alpha=0.3, label='per step')
plt.plot(trainer.history.smoothed(15), label='smoothed')
plt.xlabel('step'); plt.ylabel('loss')
plt.title('Alpamayo triplane — BEV segmentation training')
plt.legend(); plt.tight_layout(); plt.show()

/var/folders/_q/55b37d8s6sq3xq4j5ll92c400000gn/T/ipykernel_5300/3084696717.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.legend(); plt.tight_layout(); plt.show()


In [16]:
# Qualitative predictions on unseen scenes
tb = test_batches[0]
pred = trainer.predict(tb)
ext = [grid.x_bounds[0], grid.x_bounds[1], grid.y_bounds[0], grid.y_bounds[1]]

fig, ax = plt.subplots(2, 4, figsize=(12, 5))
for s in range(4):
    ax[0, s].imshow(tb.bev_labels[s].T, origin='lower', vmin=0, vmax=2, extent=ext)
    ax[0, s].set_title(f'scene {s} GT')
    ax[1, s].imshow(pred[s].T, origin='lower', vmin=0, vmax=2, extent=ext)
    ax[1, s].set_title('prediction')
    for row in range(2):
        ax[row, s].set_xlabel('x (m)'); ax[row, s].set_ylabel('y (m)')

plt.suptitle('Alpamayo BEV segmentation: GT (top) vs prediction (bottom)', y=1.01)
plt.tight_layout()
plt.show()

/var/folders/_q/55b37d8s6sq3xq4j5ll92c400000gn/T/ipykernel_5300/3933656798.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 10: Geometry verification — 90° rig rotation test 🔬

A geometry-aware lifter must map pixels → 3D **through the camera calibration**.
The test: keep images and labels the same, but rotate the extrinsics by 90°.
A properly calibrated lifter's predictions must rotate too — becoming wrong relative to
the fixed labels — so the mIoU must drop dramatically.

A geometry-free baseline (like `mlp_view`) won't care, and its score won't change.

In [17]:
import dataclasses

Rz = torch.eye(4)
Rz[0, 0], Rz[0, 1], Rz[1, 0], Rz[1, 1] = 0.0, -1.0, 1.0, 0.0  # 90 degrees about z

def rotate_rig(b):
    return dataclasses.replace(b, cameras=b.cameras.transformed(Rz))

good = trainer.evaluate(test_batches).mean_foreground()
bad  = trainer.evaluate(test_batches, transform=rotate_rig).mean_foreground()

print(f"mIoU with correct extrinsics : {good:.3f}")
print(f"mIoU with rotated extrinsics : {bad:.3f}")
print(f"→ drops by {100 * (1 - bad / max(good, 1e-9)):.0f}%  (Alpamayo uses camera geometry)")

mIoU with correct extrinsics : 0.496
mIoU with rotated extrinsics : 0.020
→ drops by 96%  (Alpamayo uses camera geometry)


## Step 11: Comparison — Alpamayo vs TPVFormer vs BEVFormer

| Property | BEVFormer | TPVFormer | **Alpamayo** |
|---|---|---|---|
| Planes | Pxy only | Pxy + Pxz + Pyz | Pxy + Pxz + Pyz |
| Scene parametrization | uniform | uniform | **Mip-NeRF 360 (non-linear)** |
| Multi-scale FPN | yes (V2) | optional | optional |
| Output | BEV map | TPV planes + BEV | TPV planes + BEV + **VLA tokens** |
| Token count (Sx=96) | ~1000+ | ~20K+ | **288** |
| Token efficiency | — | — | **72%+ fewer than patch-based** |
| Downstream use | BEV detection | 3D occupancy | **VLA planning (Alpamayo-R1)** |
| Key paper | arXiv:2203.17270 | arXiv:2302.07817 | arXiv:2506.12251 / 2511.00088 |

**When to choose Alpamayo:**
- You need a fixed-size perception token stream as input to a VLA or language model.
- Your scene has objects at varying ranges (highway driving, open parking lots).
- You want the inner region to be high-fidelity without increasing total token count.

**When TPVFormer might be better:**
- You need voxel-level 3D semantic output (occupancy prediction).
- The scene is physically bounded and the uniform grid is good enough.

In [18]:
# Quick token count comparison across methods
import matplotlib.patches as mpatches

methods = ['BEVFormer\n(V2, 4 scales)', 'TPVFormer\n(uniform)', 'Alpamayo\n(Mip-NeRF 360)']
# For the small synthetic grid (32×32×4)
bev_tokens = 32 * 32  # BEV queries
tpv_tokens = 32*32 + 32*4 + 32*4  # all three planes
alp_tokens_plane = (32 // 4)**2 + (32 // 4)*(4 // 4) + (32 // 4)*(4 // 4)  # patchified

counts = [bev_tokens, tpv_tokens, alp_tokens_plane]
colors = ['dodgerblue', 'orangered', 'mediumseagreen']

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(methods, counts, color=colors, width=0.5, alpha=0.85)
for bar, c in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 8, str(c),
            ha='center', fontsize=12, fontweight='bold')
ax.set_ylabel('Query / output tokens')
ax.set_title('Token count comparison (32×32×4 grid, patch_size=4)')
ax.set_ylim(0, max(counts) * 1.15)
plt.tight_layout()
plt.show()

/var/folders/_q/55b37d8s6sq3xq4j5ll92c400000gn/T/ipykernel_5300/3558596386.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 12: Recap, pros/cons, exercises and references

### The whole Alpamayo tokenizer in one picture

```
 N camera images ──► DINOv2 / ConvNet ──► feature maps
                                                │
          Mip-NeRF 360 non-linear grid ──► AlpamayoReferencePoints
          (denser cells near ego)               │
                                                ▼
     learned plane queries (Pxy | Pxz | Pyz) ──► [ cross-view hybrid attention
          ▲                                      [ image cross-attention (via ref. points)
          │  num_layers                          [ feed-forward
          └──────────────────────────────────────
                          │
                          ▼
                    TPVPlanes (xy, xz, yz)
                          │
         patchify_planes(patch_size=8) → 288 VLA tokens
                          │
                          ▼
         Alpamayo-R1 policy head (VLA / LLM)
```

### Pros
- **Fixed, compact token count** — compatible with any sequence-length-sensitive policy (LLMs, diffusion).
- **Non-linear grid** — more resolution near the ego where it matters most.
- **Reuses TPVFormer layers** — same well-validated deformable attention code.
- **72% token reduction** vs naive per-camera patch tokenization.

### Cons
- The non-linear grid complicates any post-processing that assumes uniform cell size.
- `patchify_planes()` with a fixed patch size can blur fine structure within each patch.
- Bounded outer region still has finite resolution; very distant objects are coarse.

---

### Exercises

1. **Change `inner_frac`**: try `0.5` and `0.9`. How does the cell spacing plot change? Does
   training mIoU change?
2. **Token budget**: for `Sx=Sy=96, Sz=48, patch_size=8`, compute the exact token count and
   verify with `patchify_planes(patch_size=8)` on a lifter built with those dimensions.
3. **Compare against TPVFormer**: run both lifters with `build_bev_model("tpvformer" | "alpamayo", ...)`
   and compare mIoU curves. Do the non-linear grid cells help?
4. **Ablate patchify**: try `patch_size=1` (no aggregation), `2`, `4`, `8`. How does the
   downstream token count trade off against information loss?
5. **Geometry-free baseline**: replace `AlpamayoReferencePoints` with `TPVReferencePoints` — does
   the training outcome change?
6. **Unbounded driving**: use a larger grid (`GridSpec(x_bounds=(-51.2, 51.2), ...)`) and note
   how the non-linear spacing helps the inner zone coverage stay dense.

---

### References

```bibtex
@article{ivanovic2025triplane,
  title   = {Efficient Multi-Camera Tokenization with Triplanes for End-to-End Driving},
  author  = {Ivanovic, Boris and others},
  journal = {arXiv:2506.12251},
  year    = {2025}
}

@article{alpamayo2025,
  title   = {Bridging Reasoning and Action Prediction for Generalizable Autonomous Driving},
  author  = {NVIDIA Research},
  journal = {arXiv:2511.00088},
  year    = {2025}
}

@inproceedings{barron2022mipnerf360,
  title     = {Mip-NeRF 360: Unbounded Anti-Aliased Neural Radiance Fields},
  author    = {Barron, Jonathan T. and Mildenhall, Ben and Verbin, Dor and Srinivasan,
               Pratul P. and Hedman, Peter},
  booktitle = {CVPR},
  year      = {2022}
}

@inproceedings{huang2023tpvformer,
  title     = {Tri-Perspective View for Vision-Based 3D Semantic Occupancy Prediction},
  author    = {Huang, Yuanhui and Zheng, Wenzhao and Zhang, Yunpeng and Zhou, Jie and
               Lu, Jiwen},
  booktitle = {CVPR},
  year      = {2023}
}
```

**Code locations:**

| Component | File |
|---|---|
| Non-linear parametrizer | `src/lifting/lifters/alpamayo/mipnerf360.py` |
| Reference points | `src/lifting/lifters/alpamayo/triplane_ref_points.py` |
| Lifter + `patchify_planes` | `src/lifting/lifters/alpamayo/alpamayo_lifter.py` |
| TPVFormer layer (shared) | `src/lifting/lifters/tpvformer/tpvformer_layer.py` |
| TPVPlanes (shared) | `src/lifting/lifters/tpvformer/tpv_planes.py` |